### Inspección y limpieza de los datos

En este seminario trabajaremos con el **Spanish Fake News Corpus**, un conjunto de noticias en español etiquetadas como **verdaderas** o **falsas** (bulos). Fue creado por investigadores de la UNAM y el IPN (México) y se ha usado en las competiciones MEX-A3T e IberLEF ([repositorio original](https://github.com/jpposadas/FakeNewsCorpusSpanish), licencia CC BY 4.0).

El corpus tiene dos partes recogidas en épocas distintas, y eso lo vamos a aprovechar:

- **Noticias de 2018** (`train.xlsx` y `development.xlsx`): las usaremos para entrenar y validar nuestros modelos.
- **Noticias de 2020-2021** (`test.xlsx`): las guardaremos como **"noticias nuevas"**, publicadas años después y con temas que no existían en 2018 (¡hola, COVID-19!). Nos servirán para comprobar si los modelos generalizan.

Este notebook está estructurado como sigue:

- **Importación de librerías**
- **BLOQUE A**: carga de los datos.
- **BLOQUE B**: inspección de los datos.
- **BLOQUE C**: limpieza y adecuación.
- **BLOQUE D**: exportación de los conjuntos resultantes.

In [ ]:
# Preparación del entorno: funciona tanto en Google Colab como en local
import os, sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and not os.path.exists("seminario-manadine-transfer-learning"):
    !git clone -q https://github.com/Vins16/seminario-manadine-transfer-learning.git
    %pip install -q -r seminario-manadine-transfer-learning/requirements.txt

if EN_COLAB:
    RAIZ = "seminario-manadine-transfer-learning"
else:
    RAIZ = "." if os.path.exists("data") else ".."
sys.path.insert(0, RAIZ)
DATA = f"{RAIZ}/data"
MODELS = f"{RAIZ}/models"

### Importación de librerías

In [ ]:
import pandas as pd
import re

### BLOQUE A: carga de los datos

Los datos vienen en ficheros Excel, así que usaremos la función de pandas que lee este formato.

In [ ]:
df_train = pd.???(f"{DATA}/raw/train.xlsx")
df_dev = pd.???(f"{DATA}/raw/development.xlsx")
df_nuevas = pd.???(f"{DATA}/raw/test.xlsx")

### BLOQUE B: inspección de los datos

El objetivo de la inspección es familiarizarnos con el conjunto de datos. Algunas preguntas que conviene hacerse:

- ¿En qué tipo de objeto están almacenados los datos?
- ¿Cuál es su dimensión?
- ¿Hay datos ausentes?
- ¿Las distintas partes del corpus tienen el mismo formato?

In [ ]:
# Tipo de objeto con el que estamos trabajando
???(df_train)

In [ ]:
# Dimensiones de cada conjunto
print(f"Entrenamiento 2018: {df_train.???[0]} filas y {df_train.???[1]} columnas")
print(f"Desarrollo 2018:    {df_dev.shape[0]} filas y {df_dev.shape[1]} columnas")
print(f"Nuevas 2020-21:     {df_nuevas.shape[0]} filas y {df_nuevas.shape[1]} columnas")

In [ ]:
# Primeras filas del conjunto de entrenamiento
df_train.???

In [ ]:
# Primeras filas de las noticias nuevas. ¿Qué diferencias ves respecto al anterior?
df_nuevas.???

Hay varias diferencias de formato que tendremos que arreglar:

1. Los nombres de las columnas están en **MAYÚSCULAS** en las noticias nuevas.
2. La etiqueta está escrita como `"Fake"`/`"True"` en 2018 y como booleano (`True`/`False`) en 2020-21.
3. Los temas están en inglés en 2018 y en español en 2020-21.

In [ ]:
# Resumen conciso de cada DataFrame: tipos de datos y valores no nulos
df_train.???

In [ ]:
df_nuevas.info()

In [ ]:
# ¿Cuántos valores ausentes hay en cada columna de las noticias nuevas?
df_nuevas.???

In [ ]:
# ¿Está equilibrada la variable objetivo?
df_train["Category"].???

In [ ]:
df_nuevas["CATEGORY"].value_counts()

In [ ]:
# ¿De qué temas hablan las noticias de cada época?
print(df_train["Topic"].value_counts(), "\n")
print(df_nuevas["TOPICS"].value_counts())

#### Cuidado con la fuga de información (*data leakage*)

Veamos qué relación hay entre la **fuente** de la noticia y su etiqueta:

In [ ]:
# Tabla de contingencia entre la fuente y la categoría
pd.???(df_train["Source"], df_train["Category"]).sort_values("Fake", ascending=False).head(10)

Algunas fuentes (webs satíricas o de bulos) publican **solo** noticias falsas. Un modelo que usara la columna `Source` acertaría casi siempre en estos datos, pero no aprendería nada útil: ante una fuente desconocida no sabría qué hacer. Por eso **trabajaremos únicamente con el texto** (titular + cuerpo de la noticia).

### BLOQUE C: limpieza y adecuación

In [ ]:
# Unimos los dos ficheros de 2018 en un único DataFrame
df_2018 = pd.???([df_train, df_dev], ignore_index=True)
df_2018.shape

In [ ]:
# Homogeneizamos los nombres de las columnas de las noticias nuevas (por ejemplo, "CATEGORY" pasa a ser "Category")
df_nuevas.columns = df_nuevas.columns.str.???

# La columna de temas se llama distinto: cambiamos "Topics" por "Topic"
df_nuevas = df_nuevas.???(columns={"Topics": "Topic"})
df_nuevas.columns

Creamos la variable objetivo `target`, que valdrá **1 si la noticia es falsa** y **0 si es verdadera**.

En las noticias nuevas, `Category == True` significa que la noticia es *verdadera*, así que hay que tener cuidado con el orden.

In [ ]:
df_2018["target"] = df_2018["Category"].???({"Fake": 1, "True": 0})
df_nuevas["target"] = df_nuevas["Category"].???({???})

print(df_2018["target"].value_counts(), "\n")
print(df_nuevas["target"].value_counts())

In [ ]:
# Traducimos los temas de 2018 al español para que ambos conjuntos sean coherentes
traduccion_temas = {
    "Politics": "Política", "Entertainment": "Entretenimiento", "Society": "Sociedad",
    "Sport": "Deporte", "Science": "Ciencia", "Health": "Salud",
    "Economy": "Economía", "Security": "Seguridad", "Education": "Educación",
}
df_2018["Topic"] = df_2018["Topic"].???
df_2018["Topic"].value_counts()

In [ ]:
# Algunas noticias nuevas no tienen titular: rellenamos los vacíos con una cadena vacía
df_nuevas["Headline"] = df_nuevas["Headline"].???
df_nuevas["Headline"].isna().sum()

Ahora limpiamos el texto. La limpieza será **ligera** a propósito: los modelos de transfer learning que usaremos en el notebook 4 se entrenaron con texto natural (con mayúsculas, tildes y signos de puntuación), así que les conviene recibirlo tal cual. La normalización más agresiva la haremos en el notebook 3, solo para el modelo clásico.

In [ ]:
def limpiar_texto(texto):
    texto = str(texto)
    texto = re.sub(r"http\S+", " ", texto)   # quitamos las URLs
    texto = re.sub(r"\s+", " ", texto)       # saltos de línea y espacios repetidos se convierten en un solo espacio
    return texto.???


limpiar_texto("  Comparte   esto:\n\nhttps://bulo.com/noticia  ¡YA!  ")

In [ ]:
for df in [df_2018, df_nuevas]:
    df["Headline"] = df["Headline"].???
    df["Text"] = df["Text"].???

#### Un detalle que se escapa fácilmente: los números

Fíjate en este fragmento de una noticia de 2018:

In [ ]:
df_2018["Text"].iloc[0][900:1100]

In [ ]:
# Proporción de noticias que contienen "*NUMBER*" y proporción que contienen algún dígito, en cada época
for nombre, df in [("2018", df_2018), ("2020-21", df_nuevas)]:
    con_number = df["Text"].str.contains("*NUMBER*", regex=False).mean()
    con_digitos = df["Text"].str.???(r"\d").mean()
    print(f"{nombre:<8} *NUMBER*: {con_number:.0%}   dígitos: {con_digitos:.0%}")

Los autores del corpus de 2018 sustituyeron cada número por la marca `*NUMBER*`, pero en las noticias de 2020-21 los números aparecen tal cual. Si no lo corregimos, el modelo podría aprender que "contener `*NUMBER*`" es una pista sobre la clase de la noticia, y esa pista no existiría nunca en las noticias nuevas. Aplicamos el mismo tratamiento a ambos conjuntos:

In [ ]:
def anonimizar_numeros(texto):
    return re.???(r"\b\d+(?:[.,]\d+)*\b", "*NUMBER*", texto)


print(anonimizar_numeros("Murieron 1.250 personas en 2020"))

df_nuevas["Headline"] = df_nuevas["Headline"].???
df_nuevas["Text"] = df_nuevas["Text"].???

Muchas noticias repiten el titular al principio del cuerpo. Creamos una única columna `texto` que contenga el titular seguido del cuerpo, evitando duplicarlo.

In [ ]:
def unir_titular(fila):
    titular, cuerpo = fila["Headline"], fila["Text"]
    if titular == "" or cuerpo.startswith(titular):
        return cuerpo
    return f"{titular}. {cuerpo}"


df_2018["texto"] = df_2018.???
df_nuevas["texto"] = df_nuevas.???
df_2018["texto"].iloc[0][:300]

In [ ]:
# ¿Hay noticias duplicadas?
print(df_2018["texto"].???.sum(), df_nuevas["texto"].duplicated().sum())

In [ ]:
# Eliminamos los duplicados para que una misma noticia no acabe a la vez en entrenamiento y validación
df_2018 = df_2018.???(subset="texto", ignore_index=True)
df_2018.shape

In [ ]:
# Longitud de cada noticia en caracteres
for df in [df_2018, df_nuevas]:
    df["longitud"] = df["texto"].???

df_2018["longitud"].describe()

In [ ]:
# ¿Tienen la misma longitud las noticias falsas (1) y las verdaderas (0)?
df_2018.???["longitud"].median()

### BLOQUE D: exportación de los conjuntos resultantes

Guardamos solo las columnas que nos serán útiles en los siguientes notebooks.

In [ ]:
columnas = ["Id", "Topic", "texto", "longitud", "target"]

df_2018[columnas].???(f"{DATA}/noticias_2018.csv", index=False)
df_nuevas[columnas].???(f"{DATA}/noticias_2020_21.csv", index=False)